In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Labs"
)

xpt_file = DATA_DIR / "TCHOL_D.xpt"
csv_file = DATA_DIR / "TCHOL_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 TCHOL_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXTC",
    "LBDTCSI"
]


if len(df) != 8086:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 3:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "TCHOL_D columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: TCHOL_D structure = "
    "8,086 rows x 3 columns."
)


# ============================================================
# 4. DEFINE LAB MEASUREMENT VARIABLES
# ============================================================

#
# CRITICAL:
#
# Both are laboratory measurements.
#
# DO NOT:
# - convert to Int64
# - round()
# - recalculate
# - trim .0
# - use float_format
#

MEASUREMENT_COLUMNS = [
    "LBXTC",
    "LBDTCSI"
]


# ============================================================
# 5. CHECK KNOWN XPORT TINY-VALUE ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    raise ValueError(
        "STOP: TCHOL_D unexpectedly contains "
        "XPORT tiny-value artifacts. "
        "Do NOT automatically replace them."
    )


print(
    "PASS: TCHOL_D contains no tiny-value artifacts."
)


# ============================================================
# 6. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
    )

    raise ValueError(
        "Unexpected zero values found. "
        "Review against CDC before proceeding."
    )


print(
    "PASS: Source contains no numeric zero values."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31128
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. CDC LBXTC CHECK
# ============================================================

s = df["LBXTC"]


print("\n--- LBXTC ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if not (
    int(s.notna().sum()) == 7360
    and
    int(s.isna().sum()) == 726
    and
    np.isclose(
        s.min(),
        78.0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        615.0,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "LBXTC differs from CDC."
    )


print(
    "PASS: LBXTC matches CDC "
    "(78-615, 7,360 valid, 726 missing)."
)


# ============================================================
# 9. CDC LBDTCSI CHECK
# ============================================================

s = df["LBDTCSI"]


print("\n--- LBDTCSI ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if not (
    int(s.notna().sum()) == 7360
    and
    int(s.isna().sum()) == 726
    and
    np.isclose(
        s.min(),
        2.02,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        15.9,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "LBDTCSI differs from CDC."
    )


print(
    "PASS: LBDTCSI matches CDC "
    "(2.02-15.9, 7,360 valid, 726 missing)."
)


# ============================================================
# 10. LAB DECIMAL BEHAVIOR
# ============================================================

print("\n--- LAB DECIMAL CHECK ---")


for col in MEASUREMENT_COLUMNS:

    values = (
        df[col]
        .dropna()
    )


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    dot_zero = sum(
        str(float(v)).endswith(".0")
        for v in values
    )


    print(
        f"{col:<10}"
        f" valid={len(values):,}"
        f" fractional={int(fractional.sum()):,}"
        f" source .0={dot_zero:,}"
    )


#
# Actual attached XPT:
#
# LBXTC
# valid       = 7,360
# fractional  = 0
# source .0   = 7,360
#
# LBDTCSI
# valid       = 7,360
# fractional  = 7,307
# source .0   = 53
#

print(
    "\nIMPORTANT:"
)

print(
    "LBXTC remains a laboratory measurement "
    "even though all released values are whole-number measurements."
)

print(
    "LBDTCSI contains genuine decimal measurements."
)


# ============================================================
# 11. DO NOT RECOMPUTE LBDTCSI
# ============================================================

#
# CDC documents:
#
# LBDTCSI = LBXTC * 0.02586
#
# But DO NOT calculate it again here.
#
# Example:
#
# source LBXTC   = 170.0
# source LBDTCSI = 4.4
#
# Released XPT value is the source of truth.
#

print(
    "\nPASS: LBDTCSI will NOT be recalculated."
)


# ============================================================
# 12. CLEAN ONLY SEQN
# ============================================================

#
# 31128.0 -> 31128
#
# Lab measurements remain untouched.
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to integer formatting."
)

print(
    "PASS: LBXTC and LBDTCSI remain untouched."
)


# ============================================================
# 13. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


#
# SEQN numeric fidelity
#

current_seqn = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    source_df["SEQN"].to_numpy(),
    current_seqn,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SEQN numeric values changed."
    )


#
# Lab measurements must be completely unchanged
#

for col in MEASUREMENT_COLUMNS:

    if not np.allclose(
        source_df[col].to_numpy(),
        df[col].to_numpy(),
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col} values changed."
        )


print(
    "PASS: All 3 variables retain "
    "their original numeric values."
)


# ============================================================
# 14. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO round()
# NO recalculation
# NO decimal trimming
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 15. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 16. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 3 CSV columns numerically "
    "match the original XPT exactly."
)


# ============================================================
# 17. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 18. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 19. SEQN RAW FORMAT CHECK
# ============================================================

bad_seqn_dot_zero = 0
bad_seqn_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    value = row["SEQN"]


    if (
        value != ""
        and
        value.endswith(".0")
    ):

        bad_seqn_dot_zero += 1


        if len(
            bad_seqn_examples
        ) < 20:

            bad_seqn_examples.append(
                (
                    row_number,
                    value
                )
            )


print(
    "\nSEQN values ending '.0':",
    bad_seqn_dot_zero
)


if bad_seqn_dot_zero != 0:

    print(
        "Examples:",
        bad_seqn_examples
    )

    raise ValueError(
        "SEQN contains unnecessary .0 formatting."
    )


print(
    "PASS: SEQN has clean integer formatting."
)


# ============================================================
# 20. STRICT RAW LAB FIDELITY
# ============================================================

#
# Compare actual CSV token directly with:
#
# str(float(original XPT value))
#
# This means:
#
# LBXTC
# 129.0 -> 129.0
#
# LBDTCSI
# 3.34 -> 3.34
#
# 4.4 -> 4.4
#
# We DO NOT force:
#
# 4.4 -> 4.40
#
# because that would be formatting rather than
# preserving pandas' decoded XPT value.
#

measurement_mismatches = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


mismatch_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            source_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_mismatches[
                col
            ] += 1


            if len(
                mismatch_examples
            ) < 20:

                mismatch_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


bad_measurements = {
    col: count
    for col, count
    in measurement_mismatches.items()
    if count > 0
}


print(
    "Lab raw-text mismatches:",
    bad_measurements
)


if bad_measurements:

    print(
        "Examples:",
        mismatch_examples
    )

    raise ValueError(
        "TCHOL laboratory formatting changed."
    )


print(
    "PASS: LBXTC and LBDTCSI "
    "appear AS IS in CSV."
)


# ============================================================
# 21. LEGITIMATE LAB .0 REPORT
# ============================================================

lab_dot_zero = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


for row in raw_rows:

    for col in MEASUREMENT_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            lab_dot_zero[
                col
            ] += 1


print(
    "\nLegitimate lab values ending '.0':"
)


for col, count in lab_dot_zero.items():

    print(
        f"{col:<10}: {count:,}"
    )


#
# Expected from attached XPT:
#
# LBXTC   = 7,360
# LBDTCSI = 53
#

if lab_dot_zero["LBXTC"] != 7360:

    raise ValueError(
        "Unexpected LBXTC .0 count."
    )


if lab_dot_zero["LBDTCSI"] != 53:

    raise ValueError(
        "Unexpected LBDTCSI .0 count."
    )


print(
    "\nPASS: Legitimate laboratory .0 "
    "representations preserved."
)


# ============================================================
# 22. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("TCHOL_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny-value corrections:",
    0
)

print(
    "SEQN:",
    "integer formatting"
)

print(
    "LBXTC:",
    "preserved AS IS"
)

print(
    "LBDTCSI:",
    "preserved AS IS"
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Raw lab differences:",
    "0 expected"
)

print(
    "SEQN .0:",
    "0 expected"
)

NHANES 2005-2006 TCHOL_D XPT -> CSV
Rows: 8,086
Columns: 3
PASS: TCHOL_D structure = 8,086 rows x 3 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: TCHOL_D contains no tiny-value artifacts.

True numeric zeros: 0
PASS: Source contains no numeric zero values.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 31128.0
Maximum: 41474.0
PASS: SEQN validated.

--- LBXTC ---
Valid: 7360
Missing: 726
Minimum: 78.0
Maximum: 615.0
PASS: LBXTC matches CDC (78-615, 7,360 valid, 726 missing).

--- LBDTCSI ---
Valid: 7360
Missing: 726
Minimum: 2.02
Maximum: 15.9
PASS: LBDTCSI matches CDC (2.02-15.9, 7,360 valid, 726 missing).

--- LAB DECIMAL CHECK ---
LBXTC      valid=7,360 fractional=0 source .0=7,360
LBDTCSI    valid=7,360 fractional=7,307 source .0=53

IMPORTANT:
LBXTC remains a laboratory measurement even though all released values are whole-number measurements.
LBDTCSI contains genuine decimal measurements.

PASS: LBDTCSI will NOT be recalculated.

PASS: SEQN converted to i